In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [3]:
df = pd.read_csv('../data/processed/dataset_long.csv')
df['datetime'] = pd.to_datetime(df['datetime'])
df.head()

,datetime,underlying_price,contract,iv,option_type,strike,hour,minute,day_of_week,date,session_progress,days_to_expiry,moneyness,log_moneyness,dist_from_atm,dist_from_atm_pct,is_ce,strike_rank,iv_neighbor_-2,iv_neighbor_-1,iv_neighbor_+1,iv_neighbor_+2,iv_neighbor_mean,wide_iv_neighbor_mean,iv_lag_1,iv_roll_mean_5,iv_roll_std_5,iv_roll_mean_10,mean_ce_iv,mean_pe_iv
0,2026-01-07 09:15:00,26111.65,NIFTY27JAN2625200CE,0.12662,CE,25200,9,15,2,2026-01-07,0.000000,20.2569,0.965086,-0.035538,911.65,0.034914,1,0.0,NaN,NaN,0.12330,0.11741,0.12330,0.11741,NaN,NaN,NaN,NaN,0.102447,0.143129
1,2026-01-07 09:20:00,26141.40,NIFTY27JAN2625200CE,0.08632,CE,25200,9,20,2,2026-01-07,0.013514,20.2535,0.963988,-0.036676,941.40,0.036012,1,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.12662,NaN,NaN,NaN,0.098723,0.147267
2,2026-01-07 09:25:00,26139.35,NIFTY27JAN2625200CE,0.09147,CE,25200,9,25,2,2026-01-07,0.027027,20.2500,0.964064,-0.036598,939.35,0.035936,1,0.0,NaN,NaN,NaN,0.09514,NaN,0.09514,0.08632,0.106470,0.028496,NaN,0.091074,0.148739
3,2026-01-07 09:30:00,26128.95,NIFTY27JAN2625200CE,0.10860,CE,25200,9,30,2,2026-01-07,0.040541,20.2465,0.964447,-0.036200,928.95,0.035553,1,0.0,NaN,NaN,0.10842,0.11150,0.10842,0.11150,0.09147,0.101470,0.021932,0.101470,0.103252,0.142787
4,2026-01-07 09:35:00,26131.90,NIFTY27JAN2625200CE,0.10462,CE,25200,9,35,2,2026-01-07,0.054054,20.2431,0.964339,-0.036313,931.90,0.035661,1,0.0,NaN,NaN,0.10538,0.12459,0.10538,0.12459,0.10860,0.103252,0.018259,0.103252,0.104097,0.147425


In [4]:
df['iv'].isna().sum()

np.int64(5460)

In [5]:
df.columns

Index(['datetime', 'underlying_price', 'contract', 'iv', 'option_type',
       'strike', 'hour', 'minute', 'day_of_week', 'date', 'session_progress',
       'days_to_expiry', 'moneyness', 'log_moneyness', 'dist_from_atm',
       'dist_from_atm_pct', 'is_ce', 'strike_rank', 'iv_neighbor_-2',
       'iv_neighbor_-1', 'iv_neighbor_+1', 'iv_neighbor_+2',
       'iv_neighbor_mean', 'wide_iv_neighbor_mean', 'iv_lag_1',
       'iv_roll_mean_5', 'iv_roll_std_5', 'iv_roll_mean_10', 'mean_ce_iv',
       'mean_pe_iv'],
      dtype='str')

In [6]:
dates = sorted(df['datetime'].dt.date.unique())

folds = [
    {'name': 'Fold 1', 'train_dates': dates[:9],  'val_dates': dates[9:11]},
    {'name': 'Fold 2', 'train_dates': dates[:10], 'val_dates': dates[10:12]},
]

In [12]:
def make_holdout(val_df):
    np.random.seed(42)

    available_rows = val_df[val_df['iv'].notna()].index

    hidden_rows = np.random.choice(available_rows,size=int(0.2*len(available_rows)),replace=False)

    ground_truth = val_df.loc[hidden_rows,'iv'].values

    val_masked = val_df.copy()

    val_masked.loc[hidden_rows,'iv'] = np.nan

    return val_masked,hidden_rows,ground_truth

In [8]:
# evaluation functions for ml models
def eval_pred(actual,predicted):
    valid = ~np.isnan(predicted)

    mse = mean_squared_error(actual[valid],predicted[valid])
    coverage = valid.mean()

    print(f"MSE: {mse:6f}")
    print(f"Coverage: {coverage:6f}")

    return mse,coverage

In [9]:
import xgboost
import lightgbm
import catboost

### Model 1: XGBoost will all features

In [10]:
from xgboost import XGBRegressor

fold_mses, fold_coverages, feature_importances = [],[],[]

drop_cols = ['iv','datetime','contract','option_type','date']

for fold in folds:
    train_df = df[df['datetime'].dt.date.isin(fold['train_dates'])].copy()

    val_df = df[df['datetime'].dt.date.isin(fold['val_dates'])].copy()

    val_masked, hidden_rows, ground_truth = make_holdout(val_df)

    # train data
    train_obs = train_df[train_df['iv'].notna()].copy()

    X_train = train_obs.drop(columns=drop_cols)
    y_train = train_obs['iv']

    # validation data
    X_val = val_masked.loc[hidden_rows].drop(columns=drop_cols)

    xgb_model = XGBRegressor(
        n_estimators= 500,
        learning_rate= 0.05,
        max_depth= 6,
        subsample= 0.8,
        colsample_bytree= 0.8,
        random_state= 2,
    )

    xgb_model.fit(X_train,y_train)
    xgb_pred = xgb_model.predict(X_val)

    xgb_mse, xgb_coverage = eval_pred(ground_truth,xgb_pred)

    fold_mses.append(xgb_mse)
    fold_coverages.append(xgb_coverage)
    feature_importances.append(xgb_model.feature_importances_)

print(f"xgb_mse: {np.mean(fold_mses):6f}")
print(f"xgb_coverage: {np.mean(fold_coverages):6f}")
print(f"feature importances: {feature_importances}")


MSE: 0.000070
Coverage: 1.000000
MSE: 0.000027
Coverage: 1.000000
xgb_mse: 0.000049
xgb_coverage: 1.000000
feature importances: [array([3.7790253e-04, 6.5848730e-03, 2.6073269e-04, 3.1602397e-04,
       4.4758245e-04, 4.6028820e-04, 3.8273609e-04, 1.2178053e-03,
       1.3457789e-03, 9.0239325e-04, 1.5925937e-03, 2.8371465e-04,
       9.9229719e-04, 1.3220002e-03, 2.3099901e-03, 3.7963623e-03,
       2.2564763e-03, 6.9269374e-02, 8.1679281e-03, 9.1375475e-04,
       1.8787275e-01, 5.5386731e-04, 7.0740551e-01, 5.5702927e-04,
       4.1024093e-04], dtype=float32), array([3.5593307e-04, 3.0944557e-03, 1.8743968e-04, 2.4557763e-04,
       4.8874796e-04, 4.1361374e-04, 4.1689051e-04, 1.2898707e-03,
       1.0478845e-03, 6.4841594e-04, 1.0786148e-03, 7.0529344e-04,
       9.3577954e-04, 1.3852600e-03, 1.9683843e-03, 3.3845291e-03,
       1.8066594e-03, 6.0578205e-02, 7.7353991e-03, 1.0245242e-03,
       2.3247994e-01, 5.5427814e-04, 6.7727518e-01, 5.1713292e-04,
       3.8206877e-04], dtype

In [17]:
avg_importance = np.mean(feature_importances,axis=0)

importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': avg_importance
})

importance_df = importance_df.sort_values(by='importance',ascending=False)

importance_df

,feature,importance
22,iv_roll_mean_10,0.692340
20,iv_roll_mean_5,0.210176
17,iv_neighbor_mean,0.064924
18,wide_iv_neighbor_mean,0.007952
1,strike,0.004840
15,iv_neighbor_+1,0.003590
14,iv_neighbor_-1,0.002139
16,iv_neighbor_+2,0.002032
13,iv_neighbor_-2,0.001354
10,dist_from_atm_pct,0.001336


In [18]:
selected_features = importance_df[importance_df['importance']>=0.001]['feature'].tolist()

selected_features

['iv_roll_mean_10',
 'iv_roll_mean_5',
 'iv_neighbor_mean',
 'wide_iv_neighbor_mean',
 'strike',
 'iv_neighbor_+1',
 'iv_neighbor_-1',
 'iv_neighbor_+2',
 'iv_neighbor_-2',
 'dist_from_atm_pct',
 'moneyness',
 'log_moneyness']

In [19]:
df_pruned = df[selected_features+['iv','contract','option_type','date','datetime']].copy()
print(df_pruned.shape)
df_pruned.head()

(27300, 17)


,iv_roll_mean_10,iv_roll_mean_5,iv_neighbor_mean,wide_iv_neighbor_mean,strike,iv_neighbor_+1,iv_neighbor_-1,iv_neighbor_+2,iv_neighbor_-2,dist_from_atm_pct,moneyness,log_moneyness,iv,contract,option_type,date,datetime
0,NaN,NaN,0.12330,0.11741,25200,0.12330,NaN,0.11741,NaN,0.034914,0.965086,-0.035538,0.12662,NIFTY27JAN2625200CE,CE,2026-01-07,2026-01-07 09:15:00
1,NaN,NaN,NaN,NaN,25200,NaN,NaN,NaN,NaN,0.036012,0.963988,-0.036676,0.08632,NIFTY27JAN2625200CE,CE,2026-01-07,2026-01-07 09:20:00
2,NaN,0.106470,NaN,0.09514,25200,NaN,NaN,0.09514,NaN,0.035936,0.964064,-0.036598,0.09147,NIFTY27JAN2625200CE,CE,2026-01-07,2026-01-07 09:25:00
3,0.101470,0.101470,0.10842,0.11150,25200,0.10842,NaN,0.11150,NaN,0.035553,0.964447,-0.036200,0.10860,NIFTY27JAN2625200CE,CE,2026-01-07,2026-01-07 09:30:00
4,0.103252,0.103252,0.10538,0.12459,25200,0.10538,NaN,0.12459,NaN,0.035661,0.964339,-0.036313,0.10462,NIFTY27JAN2625200CE,CE,2026-01-07,2026-01-07 09:35:00


In [20]:
df_pruned['iv'].isna().sum()

np.int64(5460)

In [21]:
df_pruned.to_csv('../data/processed/dataset_long_pruned.csv',index=False)

### Model 2: XGBoost with pruned features

In [22]:
fold_mses, fold_coverages = [],[]

drop_cols = ['iv','datetime','contract','option_type','date']

for fold in folds:
    train_df = df_pruned[df_pruned['datetime'].dt.date.isin(fold['train_dates'])].copy()

    val_df = df_pruned[df_pruned['datetime'].dt.date.isin(fold['val_dates'])].copy()

    val_masked, hidden_rows, ground_truth = make_holdout(val_df)

    # train data
    train_obs = train_df[train_df['iv'].notna()].copy()

    X_train = train_obs.drop(columns=drop_cols)
    y_train = train_obs['iv']

    # validation data
    X_val = val_masked.loc[hidden_rows].drop(columns=drop_cols)

    xgb_model = XGBRegressor(
        n_estimators= 500,
        learning_rate= 0.05,
        max_depth= 6,
        subsample= 0.8,
        colsample_bytree= 0.8,
        random_state= 2,
    )

    xgb_model.fit(X_train,y_train)
    xgb_pred = xgb_model.predict(X_val)

    xgb_mse, xgb_coverage = eval_pred(ground_truth,xgb_pred)

    fold_mses.append(xgb_mse)
    fold_coverages.append(xgb_coverage)

print(f"xgbp_mse: {np.mean(fold_mses):6f}")
print(f"xgbp_coverage: {np.mean(fold_coverages):6f}")

MSE: 0.000067
Coverage: 1.000000
MSE: 0.000023
Coverage: 1.000000
xgbp_mse: 0.000045
xgbp_coverage: 1.000000


### Model 3: LightGBM

In [23]:
from lightgbm import LGBMRegressor

fold_mses, fold_coverages = [],[]

drop_cols = ['iv','datetime','contract','option_type','date']

for fold in folds:
    train_df = df_pruned[df_pruned['datetime'].dt.date.isin(fold['train_dates'])].copy()

    val_df = df_pruned[df_pruned['datetime'].dt.date.isin(fold['val_dates'])].copy()

    val_masked, hidden_rows, ground_truth = make_holdout(val_df)

    # train data
    train_obs = train_df[train_df['iv'].notna()].copy()

    X_train = train_obs.drop(columns=drop_cols)
    y_train = train_obs['iv']

    # validation data
    X_val = val_masked.loc[hidden_rows].drop(columns=drop_cols)

    lgbm_model = LGBMRegressor(
        n_estimators= 500,
        learning_rate= 0.05,
        max_depth= 6,
        subsample= 0.8,
        colsample_bytree= 0.8,
        random_state= 2,
    )

    lgbm_model.fit(X_train,y_train)
    lgbm_pred = lgbm_model.predict(X_val)

    lgbm_mse, lgbm_coverage = eval_pred(ground_truth,lgbm_pred)

    fold_mses.append(lgbm_mse)
    fold_coverages.append(lgbm_coverage)

print(f"MSE from both folds: {fold_mses}")
print(f"lgbm_mse: {np.mean(fold_mses):6f}")
print(f"lgbm_coverage: {np.mean(fold_coverages):6f}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004757 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2834
[LightGBM] [Info] Number of data points in the train set: 15089, number of used features: 12
[LightGBM] [Info] Start training from score 0.129295
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,

### Model 4: CatBoost

In [24]:
from catboost import CatBoostRegressor

fold_mses, fold_coverages = [],[]

drop_cols = ['iv','datetime','contract','option_type','date']

for fold in folds:
    train_df = df_pruned[df_pruned['datetime'].dt.date.isin(fold['train_dates'])].copy()

    val_df = df_pruned[df_pruned['datetime'].dt.date.isin(fold['val_dates'])].copy()

    val_masked, hidden_rows, ground_truth = make_holdout(val_df)

    # train data
    train_obs = train_df[train_df['iv'].notna()].copy()

    X_train = train_obs.drop(columns=drop_cols)
    y_train = train_obs['iv']

    # validation data
    X_val = val_masked.loc[hidden_rows].drop(columns=drop_cols)

    cb_model = CatBoostRegressor(
        n_estimators= 500,
        learning_rate= 0.05,
        max_depth= 6,
        subsample= 0.8,
        random_state= 2,
    )

    cb_model.fit(X_train,y_train)
    cb_pred = cb_model.predict(X_val)

    cb_mse, cb_coverage = eval_pred(ground_truth,cb_pred)

    fold_mses.append(cb_mse)
    fold_coverages.append(cb_coverage)

print(f"MSE from both folds: {fold_mses}")
print(f"cb_mse: {np.mean(fold_mses):6f}")
print(f"cb_coverage: {np.mean(fold_coverages):6f}")

0:	learn: 0.0285783	total: 173ms	remaining: 1m 26s
1:	learn: 0.0272713	total: 185ms	remaining: 46.1s
2:	learn: 0.0260105	total: 195ms	remaining: 32.2s
3:	learn: 0.0248180	total: 202ms	remaining: 25s
4:	learn: 0.0236856	total: 212ms	remaining: 21s
5:	learn: 0.0226357	total: 219ms	remaining: 18s
6:	learn: 0.0216363	total: 225ms	remaining: 15.9s
7:	learn: 0.0206699	total: 233ms	remaining: 14.3s
8:	learn: 0.0197516	total: 242ms	remaining: 13.2s
9:	learn: 0.0188914	total: 251ms	remaining: 12.3s
10:	learn: 0.0180568	total: 258ms	remaining: 11.5s
11:	learn: 0.0172762	total: 265ms	remaining: 10.8s
12:	learn: 0.0165296	total: 272ms	remaining: 10.2s
13:	learn: 0.0158327	total: 278ms	remaining: 9.65s
14:	learn: 0.0151552	total: 284ms	remaining: 9.18s
15:	learn: 0.0145105	total: 291ms	remaining: 8.81s
16:	learn: 0.0138960	total: 298ms	remaining: 8.48s
17:	learn: 0.0133114	total: 305ms	remaining: 8.15s
18:	learn: 0.0127474	total: 310ms	remaining: 7.85s
19:	learn: 0.0122314	total: 315ms	remaining: 7